In [ ]:
import datetime
import awswrangler as wr
import mlflow
import optuna

from mlflow.models import infer_signature
from mlflow import MlflowClient
from mlflow_aux import get_or_create_experiment
from optuna_aux import champion_callback

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Para que Optuna sea menos verboso
optuna.logging.set_verbosity(optuna.logging.ERROR)

# ---------------------------------------------------------
# Carga de datos
# ---------------------------------------------------------
X_train = wr.s3.read_csv("s3://data/final/train/dielectron_X_train.csv")
y_train = wr.s3.read_csv("s3://data/final/train/dielectron_y_train.csv")

X_test = wr.s3.read_csv("s3://data/final/test/dielectron_X_test.csv")
y_test = wr.s3.read_csv("s3://data/final/test/dielectron_y_test.csv")

# ---------------------------------------------------------
# Funciones auxiliares
# ---------------------------------------------------------
def build_regressor(params):
    if params["model"] == "GradientBoosting":
        return GradientBoostingRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            learning_rate=params["learning_rate"],
            subsample=params["subsample"],
            random_state=42,
        )
    if params["model"] == "RandomForest":
        return RandomForestRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            max_features=params["max_features"],
            n_jobs=-1,
            random_state=42,
        )
    if params["model"] == "Ridge":
        return Ridge(alpha=params["alpha"], random_state=42)
    raise ValueError(f"Modelo no soportado: {params['model']}")

def objective(trial, X_train, y_train, experiment_id):
    with mlflow.start_run(
        experiment_id=experiment_id,
        run_name=f"trial_{trial.number}",
        nested=True,
    ):
        model_type = trial.suggest_categorical(
            "model", ["GradientBoosting", "RandomForest", "Ridge"]
        )

        params = {"model": model_type}
        if model_type == "GradientBoosting":
            params.update(
                {
                    "n_estimators": trial.suggest_int("gb_n_estimators", 50, 250),
                    "max_depth": trial.suggest_int("gb_max_depth", 3, 10),
                    "learning_rate": trial.suggest_float(
                        "gb_learning_rate", 0.01, 0.3, log=True
                    ),
                    "subsample": trial.suggest_float("gb_subsample", 0.6, 1.0),
                }
            )
        elif model_type == "RandomForest":
            params.update(
                {
                    "n_estimators": trial.suggest_int("rf_n_estimators", 50, 250),
                    "max_depth": trial.suggest_int("rf_max_depth", 4, 20),
                    "max_features": trial.suggest_categorical(
                        "rf_max_features", ["sqrt", "log2", 0.8]
                    ),
                }
            )
        else:
            params.update(
                {
                    "alpha": trial.suggest_float("ridge_alpha", 1e-3, 10.0, log=True),
                }
            )

        model = build_regressor(params)

        scores = cross_val_score(
            model,
            X_train,
            y_train.values.ravel(),
            cv=5,
            scoring="neg_root_mean_squared_error",
            n_jobs=-1,
        )
        rmse = -scores.mean()

        mlflow.log_params(params)
        mlflow.log_metric("cv_rmse", rmse)
        mlflow.log_param("trial_number", trial.number)

    return rmse

# ---------------------------------------------------------
# Experimento MLflow
# ---------------------------------------------------------
experiment_id = get_or_create_experiment("Dielectron Mass Regression")
print("MLflow experiment_id:", experiment_id)

run_name_parent = "best_hyperparam_dielectron_" + datetime.datetime.now().strftime(
    "%Y%m%d-%H%M%S"
)

with mlflow.start_run(
    experiment_id=experiment_id,
    run_name=run_name_parent,
    nested=False,
):
    study = optuna.create_study(direction="minimize")
    study.optimize(
        lambda trial: objective(trial, X_train, y_train, experiment_id),
        n_trials=80,
        callbacks=[champion_callback],
    )

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_cv_rmse", study.best_value)

    mlflow.set_tags(
        {
            "project": "Dielectron Mass Regression",
            "optimizer_engine": "optuna",
            "model_family": "sklearn",
            "target": "M",
        }
    )

    # ---------------------------------------------------------
    # Entrenamos el mejor modelo
    # ---------------------------------------------------------
    best_model = build_regressor(study.best_params)
    best_model.fit(X_train, y_train.values.ravel())

    # ---------------------------------------------------------
    # Evaluamos en test
    # ---------------------------------------------------------
    y_pred = best_model.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_mae = mean_absolute_error(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)

    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_r2", test_r2)

    # ---------------------------------------------------------
    # Guardamos el modelo en MLflow
    # ---------------------------------------------------------
    artifact_path = "model"
    signature = infer_signature(X_train, best_model.predict(X_train))

    mlflow.sklearn.log_model(
        sk_model=best_model,
        artifact_path=artifact_path,
        signature=signature,
        serialization_format="cloudpickle",
        registered_model_name="dielectron_mass_regressor_dev",
    )

    model_uri = mlflow.get_artifact_uri(artifact_path)

    # ---------------------------------------------------------
    # Registramos la versión productiva
    # ---------------------------------------------------------
    client = MlflowClient()
    prod_name = "dielectron_mass_regressor_prod"
    try:
        client.create_registered_model(
            name=prod_name,
            description="Regressor for dielectron invariant mass",
        )
    except Exception:
        pass

    result = client.create_model_version(
        name=prod_name,
        source=model_uri,
        run_id=mlflow.active_run().info.run_id,
        tags={
            "model": type(best_model).__name__,
            "test_rmse": str(test_rmse),
            "test_r2": str(test_r2),
        },
    )

    client.set_registered_model_alias(prod_name, "champion", result.version)

    print("Best params:", study.best_params)
    print(f"Test RMSE: {test_rmse:.4f}")
    print(f"Test MAE : {test_mae:.4f}")
    print(f"Test R2  : {test_r2:.6f}")
    print("Model URI:", model_uri)

In [ ]:
import datetime
import time
import awswrangler as wr
import mlflow
import optuna

from mlflow.models import infer_signature
from mlflow import MlflowClient
from mlflow_aux import get_or_create_experiment
from optuna_aux import champion_callback

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# Para que Optuna sea menos verboso
optuna.logging.set_verbosity(optuna.logging.ERROR)

# ---------------------------------------------------------
# Carga de datos
# ---------------------------------------------------------
X_train = wr.s3.read_csv("s3://data/final/train/dielectron_X_train.csv")
y_train = wr.s3.read_csv("s3://data/final/train/dielectron_y_train.csv")

X_test = wr.s3.read_csv("s3://data/final/test/dielectron_X_test.csv")
y_test = wr.s3.read_csv("s3://data/final/test/dielectron_y_test.csv")

# ---------------------------------------------------------
# Funciones auxiliares
# ---------------------------------------------------------
def build_regressor(params):
    if params["model"] == "GradientBoosting":
        return GradientBoostingRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            learning_rate=params["learning_rate"],
            subsample=params["subsample"],
            random_state=42,
        )
    if params["model"] == "RandomForest":
        return RandomForestRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            max_features=params["max_features"],
            n_jobs=-1,
            random_state=42,
        )
    if params["model"] == "Ridge":
        return Ridge(alpha=params["alpha"], random_state=42)
    raise ValueError(f"Modelo no soportado: {params['model']}")

def objective(trial, X_train, y_train, experiment_id):
    with mlflow.start_run(
        experiment_id=experiment_id,
        run_name=f"trial_{trial.number}",
        nested=True,
    ):
        model_type = trial.suggest_categorical(
            "model", ["GradientBoosting", "RandomForest", "Ridge"]
        )

        params = {"model": model_type}
        if model_type == "GradientBoosting":
            params.update(
                {
                    "n_estimators": trial.suggest_int("gb_n_estimators", 50, 250),
                    "max_depth": trial.suggest_int("gb_max_depth", 3, 10),
                    "learning_rate": trial.suggest_float(
                        "gb_learning_rate", 0.01, 0.3, log=True
                    ),
                    "subsample": trial.suggest_float("gb_subsample", 0.6, 1.0),
                }
            )
        elif model_type == "RandomForest":
            params.update(
                {
                    "n_estimators": trial.suggest_int("rf_n_estimators", 50, 250),
                    "max_depth": trial.suggest_int("rf_max_depth", 4, 20),
                    "max_features": trial.suggest_categorical(
                        "rf_max_features", ["sqrt", "log2", 0.8]
                    ),
                }
            )
        else:
            params.update(
                {
                    "alpha": trial.suggest_float("ridge_alpha", 1e-3, 10.0, log=True),
                }
            )

        model = build_regressor(params)

        scores = cross_val_score(
            model,
            X_train,
            y_train.values.ravel(),
            cv=5,
            scoring="neg_root_mean_squared_error",
            n_jobs=-1,
        )
        rmse = -scores.mean()

        mlflow.log_params(params)
        mlflow.log_metric("cv_rmse", rmse)
        mlflow.log_param("trial_number", trial.number)

    return rmse

def calculate_regression_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(
        np.abs((y_true.values.ravel() - y_pred) / np.maximum(np.abs(y_true.values.ravel()), 1e-8))
    ) * 100
    return rmse, mae, r2, mape

def calculate_classification_metrics(y_true, y_pred, y_proba=None):
    metrics = {
        "accuracy": None,
        "precision": None,
        "recall": None,
        "f1": None,
        "auc": None,
        "log_loss": None,
    }

    y_true_flat = y_true.values.ravel()
    unique_values = np.unique(y_true_flat)

    if len(unique_values) == 2 and set(unique_values).issubset({0, 1}):
        y_pred_bin = np.round(y_pred).astype(int)
        metrics["accuracy"] = accuracy_score(y_true_flat, y_pred_bin)
        metrics["precision"] = precision_score(y_true_flat, y_pred_bin, zero_division=0)
        metrics["recall"] = recall_score(y_true_flat, y_pred_bin, zero_division=0)
        metrics["f1"] = f1_score(y_true_flat, y_pred_bin, zero_division=0)

        if y_proba is not None:
            try:
                proba = y_proba[:, 1] if y_proba.ndim == 2 else y_proba
                metrics["auc"] = roc_auc_score(y_true_flat, proba)
                metrics["log_loss"] = log_loss(y_true_flat, proba)
            except Exception:
                metrics["auc"] = None
                metrics["log_loss"] = None

    return metrics

# ---------------------------------------------------------
# Experimento MLflow
# ---------------------------------------------------------
experiment_id = get_or_create_experiment("Dielectron Mass Regression")
print("MLflow experiment_id:", experiment_id)

run_name_parent = "best_hyperparam_dielectron_" + datetime.datetime.now().strftime(
    "%Y%m%d-%H%M%S"
)

with mlflow.start_run(
    experiment_id=experiment_id,
    run_name=run_name_parent,
    nested=False,
):
    study = optuna.create_study(direction="minimize")
    study.optimize(
        lambda trial: objective(trial, X_train, y_train, experiment_id),
        n_trials=80,
        callbacks=[champion_callback],
    )

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_cv_rmse", study.best_value)
    mlflow.set_tags(
        {
            "project": "Dielectron Mass Regression",
            "optimizer_engine": "optuna",
            "model_family": "sklearn",
            "target": "M",
        }
    )

    # ---------------------------------------------------------
    # Entrenamos el mejor modelo
    # ---------------------------------------------------------
    train_start = time.time()
    best_model = build_regressor(study.best_params)
    best_model.fit(X_train, y_train.values.ravel())
    train_end = time.time()

    y_train_pred = best_model.predict(X_train)
    (
        train_rmse,
        train_mae,
        train_r2,
        train_mape,
    ) = calculate_regression_metrics(y_train, y_train_pred)
    train_clf = calculate_classification_metrics(y_train, y_train_pred)

    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("train_mape", train_mape)
    mlflow.log_metric("training_time", train_end - train_start)
    mlflow.log_metric("train_sample_count", len(X_train))

    for name, value in train_clf.items():
        if value is not None:
            mlflow.log_metric(f"train_{name}", value)

    # ---------------------------------------------------------
    # Evaluamos en test
    # ---------------------------------------------------------
    test_start = time.time()
    y_pred = best_model.predict(X_test)
    test_end = time.time()

    test_rmse, test_mae, test_r2, test_mape = calculate_regression_metrics(
        y_test, y_pred
    )
    test_clf = calculate_classification_metrics(y_test, y_pred)

    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("test_mape", test_mape)
    mlflow.log_metric("inference_time", test_end - test_start)
    mlflow.log_metric("test_sample_count", len(X_test))

    for name, value in test_clf.items():
        if value is not None:
            mlflow.log_metric(f"test_{name}", value)

    # ---------------------------------------------------------
    # Guardamos el modelo en MLflow
    # ---------------------------------------------------------
    artifact_path = "model"
    signature = infer_signature(X_train, best_model.predict(X_train))

    mlflow.sklearn.log_model(
        sk_model=best_model,
        artifact_path=artifact_path,
        signature=signature,
        serialization_format="cloudpickle",
        registered_model_name="dielectron_mass_regressor_dev",
    )

    model_uri = mlflow.get_artifact_uri(artifact_path)

    # ---------------------------------------------------------
    # Registramos la versión productiva
    # ---------------------------------------------------------
    client = MlflowClient()
    prod_name = "dielectron_mass_regressor_prod"
    try:
        client.create_registered_model(
            name=prod_name,
            description="Regressor for dielectron invariant mass",
        )
    except Exception:
        pass

    result = client.create_model_version(
        name=prod_name,
        source=model_uri,
        run_id=mlflow.active_run().info.run_id,
        tags={
            "model": type(best_model).__name__,
            "test_rmse": str(test_rmse),
            "test_r2": str(test_r2),
        },
    )

    client.set_registered_model_alias(prod_name, "champion", result.version)

    print("Best params:", study.best_params)
    print(f"Test RMSE: {test_rmse:.4f}")
    print(f"Test MAE : {test_mae:.4f}")
    print(f"Test R2  : {test_r2:.6f}")
    print("Model URI:", model_uri)